In [85]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
model = LinearRegression()
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score

df = pd.read_csv("../data/hour.csv")

df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [86]:
df["dteday"] = pd.to_datetime(df["dteday"])

df["year"] = df["dteday"].dt.year
df["month"] = df["dteday"].dt.month
df["day"] = df["dteday"].dt.day
df["dayofweek"] = df["dteday"].dt.day_of_week

df["isweekend"] = (df["dayofweek"] >= 5).astype(int)

print(df[["dteday", "year", "month", "day", "dayofweek", "isweekend"]].head())

      dteday  year  month  day  dayofweek  isweekend
0 2011-01-01  2011      1    1          5          1
1 2011-01-01  2011      1    1          5          1
2 2011-01-01  2011      1    1          5          1
3 2011-01-01  2011      1    1          5          1
4 2011-01-01  2011      1    1          5          1


In [87]:
df["hr_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
df["hr_cos"] = np.cos(2 * np.pi * df["hr"] / 24)

df[["hr", "hr_sin", "hr_cos"]].head(10)

,hr,hr_sin,hr_cos
0,0,0.000000,1.000000e+00
1,1,0.258819,9.659258e-01
2,2,0.500000,8.660254e-01
3,3,0.707107,7.071068e-01
4,4,0.866025,5.000000e-01
5,5,0.965926,2.588190e-01
6,6,1.000000,6.123234e-17
7,7,0.965926,-2.588190e-01
8,8,0.866025,-5.000000e-01
9,9,0.707107,-7.071068e-01


In [88]:
weather_encoded = pd.get_dummies(
    df["weathersit"], prefix="weather", drop_first=True, dtype=int
)

season_encoded = pd.get_dummies(
    df["season"], prefix="season", drop_first=True, dtype=int
)

df = pd.concat([df, weather_encoded, season_encoded], axis=1)

df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,...,dayofweek,isweekend,hr_sin,hr_cos,weather_2,weather_3,weather_4,season_2,season_3,season_4
0,1,2011-01-01,1,0,1,0,0,6,0,1,...,5,1,0.000000,1.000000,0,0,0,0,0,0
1,2,2011-01-01,1,0,1,1,0,6,0,1,...,5,1,0.258819,0.965926,0,0,0,0,0,0
2,3,2011-01-01,1,0,1,2,0,6,0,1,...,5,1,0.500000,0.866025,0,0,0,0,0,0
3,4,2011-01-01,1,0,1,3,0,6,0,1,...,5,1,0.707107,0.707107,0,0,0,0,0,0
4,5,2011-01-01,1,0,1,4,0,6,0,1,...,5,1,0.866025,0.500000,0,0,0,0,0,0


In [89]:
print("duplicate_columns :=", df.columns.duplicated().sum())
print(df.columns[df.columns.duplicated()].tolist())

duplicate_columns := 0
[]


### Feature Engineering Insight
Feature Engineering transforms raw data into meaningful features using date-time extraction, weekend identification, cyclic encoding, and categorical encoding to improve machine learning model learning.

In [90]:
df.columns

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt', 'year', 'month', 'day', 'dayofweek',
       'isweekend', 'hr_sin', 'hr_cos', 'weather_2', 'weather_3', 'weather_4',
       'season_2', 'season_3', 'season_4'],
      dtype='str')

instant, dteday, casual, registerd

In [91]:
x = df[[ 'hr', 'holiday',
       'workingday', 'temp', 'atemp', 'hum', 'windspeed',
         'year', 'month', 'day', 'dayofweek',
       'isweekend', 'hr_sin', 'hr_cos', 'weather_2', 'weather_3',
       'weather_4', 'season_2', 'season_3', 'season_4']]

y = df['cnt']

print("x", x.shape)
print("y", y.shape)
print("features", x.columns.tolist())

x (17379, 20)
y (17379,)
features ['hr', 'holiday', 'workingday', 'temp', 'atemp', 'hum', 'windspeed', 'year', 'month', 'day', 'dayofweek', 'isweekend', 'hr_sin', 'hr_cos', 'weather_2', 'weather_3', 'weather_4', 'season_2', 'season_3', 'season_4']


In [92]:
x.isnull().sum().sum(), y.isnull().sum()


(np.int64(0), np.int64(0))

Feature Selection: Selected 22 features for model training and separated cnt as the target variable. Removed ID, redundant date columns, and data leakage columns.

In [93]:
split_index = int(len(x) * 0.8)

x_train = x.iloc[:split_index]
x_test = x.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("x train:", x_train.shape)
print("x test:", x_test.shape)
print("y train:", y_train.shape)
print("y test:", y_test.shape)

x train: (13903, 20)
x test: (3476, 20)
y train: (13903,)
y test: (3476,)


Train-Test Split: Split the dataset into 80% training data and 20% testing data using scikit-learn. Used random_state=42 for reproducibility.

In [94]:
model.fit(x_train, y_train)

prediction = model.predict(x_test)

print("First 10 predictions:")
print(prediction[:10])

First 10 predictions:
[373.65183497 388.94735395 409.5543432  420.9364492  331.20585479
 381.45257733 359.13432064 338.93797417 307.81901186 281.90423755]


In [95]:
feature_importance = pd.DataFrame({
    "feature" : x_train.columns,
    "coefficient" : model.coef_
})

feature_importance

,feature,coefficient
0,hr,2.722969
1,holiday,-11.499104
2,workingday,8.947360
3,temp,125.496557
4,atemp,77.361266
5,hum,-39.371007
6,windspeed,-23.392096
7,year,84.408981
8,month,2.552898
9,day,-0.083214


In [96]:
comparison = pd.DataFrame({
    "actual" : y_test,
    "predicted" : prediction
})

comparison.head(10)

,actual,predicted
13903,283,373.651835
13904,253,388.947354
13905,261,409.554343
13906,306,420.936449
13907,445,331.205855
13908,868,381.452577
13909,814,359.134321
13910,610,338.937974
13911,448,307.819012
13912,317,281.904238


In [97]:
mae = mean_absolute_error(y_test, prediction)

print("MAE:", mae)

MAE: 122.40663429364251


In [98]:
mse = mean_squared_error(y_test, prediction)
print("Mean Squared Error:", mse)

Mean Squared Error: 27191.903005408585


In [99]:
rmse = np.sqrt(mse)
print("Root Mean Squared Error:", rmse)

Root Mean Squared Error: 164.89967557702647


In [100]:
r2 = r2_score(y_test, prediction)
print("R-squared:", r2)

R-squared: 0.4406518254554438


### Model Evaluation Insight
The Linear Regression model achieved an R² score of 0.5112, explaining approximately 51.12% of the target variation. The MAE was 91.59 and RMSE was 124.41, indicating noticeable prediction errors.

In [101]:
print(df[[
    "weather_1", "weather_2", "weather_3", "weather_4"
]].sum(axis=1).value_counts())

print(df[[
    "season_1", "season_2", "season_3", "season_4"
]].sum(axis=1).value_counts())

KeyError: "['weather_1'] not in index"

In [ ]:
print(df.columns.tolist())